In [1]:
# Import the necessary libraries
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

from datasets import load_dataset

import warnings
warnings.filterwarnings('ignore')

# Data Preprocessing

In [2]:
# Load datasets
passage_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")
test_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")

In [3]:
# Check the dataframes
print(passage_df.columns)
print(test_df.columns)
passage_df.head()

Index(['passage'], dtype='object')
Index(['question', 'answer', 'relevant_passage_ids'], dtype='object')


,passage
id,
9797,New data on viruses isolated from patients wit...
11906,We describe an improved method for detecting d...
16083,We have studied the effects of curare on respo...
23188,Kinetic and electrophoretic properties of 230-...
23469,Male Wistar specific-pathogen-free rats aged 2...


In [4]:
passage_df.shape

(40221, 1)

In [5]:
# Check test dataframe
test_df.head()

,question,answer,relevant_passage_ids
id,,,
0,Is Hirschsprung disease a mendelian or a multi...,"Coding sequence mutations in RET, GDNF, EDNRB,...","[20598273, 6650562, 15829955, 15617541, 230011..."
1,List signaling molecules (ligands) that intera...,The 7 known EGFR ligands are: epidermal growt...,"[23821377, 24323361, 23382875, 22247333, 23787..."
2,Is the protein Papilin secreted?,"Yes, papilin is a secreted protein","[21784067, 19297413, 15094122, 7515725, 332004..."
3,Are long non coding RNAs spliced?,Long non coding RNAs appear to be spliced thro...,"[22955974, 21622663, 22707570, 22955988, 24285..."
4,Is RANKL secreted from the cells?,Receptor activator of nuclear factor κB ligand...,"[22867712, 23827649, 21618594, 23835909, 24265..."


In [6]:
test_df.shape

(4719, 3)

## Drop missing values

In [7]:
if 'text' in passage_df.columns:
    passage_df.dropna(subset=['text'], inplace=True)
if 'question' in test_df.columns:
    test_df.dropna(subset=['question'], inplace=True)

## Cleaning Function

In [8]:
def clean_text(text):
    if isinstance(text, str):
        text = text.lower()
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'<.*?>', '', text)  # Remove HTML tags if any
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters
        text = text.strip()
    return text

if 'passage' in passage_df.columns:
    passage_df['clean_passage'] = passage_df['passage'].apply(clean_text)
if 'question' in test_df.columns:
    test_df['clean_question'] = test_df['question'].apply(clean_text)
if 'answer' in test_df.columns:
    test_df['clean_answer'] = test_df['answer'].apply(clean_text)

## Chunking for retrieval

In [9]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
MAX_TOKENS = min(256, tokenizer.model_max_length)
OVERLAP = 40

def chunk_passage(text, max_tokens=MAX_TOKENS, overlap=OVERLAP):
    tokens = tokenizer.tokenize(text)
    chunks = []
    idx = 0
    while idx < len(tokens):
        chunk = tokens[idx : idx + max_tokens]
        chunk = tokenizer.convert_tokens_to_string(chunk)
        chunks.append(chunk)
        idx += max_tokens - overlap
    return chunks

In [10]:
if 'clean_passage' in passage_df.columns:
    passage_df['chunks'] = passage_df['clean_passage'].apply(lambda txt: chunk_passage(txt) if pd.notnull(txt) else [])

Token indices sequence length is longer than the specified maximum sequence length for this model (557 > 512). Running this sequence through the model will result in indexing errors


## Flatten to dataframe

In [11]:
chunked_passages = []
for idx, row in passage_df.iterrows():
    if isinstance(row['chunks'], list):
        for c_idx, chunk in enumerate(row['chunks']):
            chunked_passages.append({
                'doc_id': row.name,
                'chunk_id': f"{row.name}_{c_idx}",
                'chunk': chunk
            })
chunked_df = pd.DataFrame(chunked_passages)

## QA pairs for transformer finetuning

In [12]:
if {'clean_question', 'clean_answer'}.issubset(test_df.columns):
    qa_pairs = test_df[['clean_question', 'clean_answer']].dropna()
    qa_pairs = qa_pairs.rename(columns={'clean_question': 'question', 'clean_answer': 'answer'})
else:
    qa_pairs = pd.DataFrame(columns=['question', 'answer'])

## Train/validation split

In [13]:
if len(qa_pairs) > 1:
    train_set, val_set = train_test_split(qa_pairs, test_size=0.2, random_state=42)
else:
    train_set, val_set = qa_pairs, qa_pairs

In [14]:
# Save to disk
chunked_df.to_csv("chunked_passages.csv", index=False)
train_set.to_csv("train_set.csv", index=False)
val_set.to_csv("val_set.csv", index=False)

In [16]:
# Save tokenizer
tokenizer.save_pretrained("./bert-base-uncased-tokenizer")
print("Tokenizer saved")

Tokenizer saved


## Summary

In [15]:
print(f"Total passage chunks: {len(chunked_df)}")
print(f"Training examples: {len(train_set)} | Validation examples: {len(val_set)}")

Total passage chunks: 64395
Training examples: 3775 | Validation examples: 944


The train_set and val_set were split only from QA pairs extracted from `test_df`, since only it had both question and answers fields.

The `passage_df` lacks complete QA info, so including it would produce empty or invalid examples—not useful for training.